In [1]:
import sys; sys.path.append("..")
from src.markov import efecto_remocion
print(efecto_remocion)

<function efecto_remocion at 0x1085677f0>


In [3]:
objetivo = leads["conv_90"].mean()

def simular(seed, N=20_000, escala=1.0):
    rng = np.random.default_rng(seed)
    hz = haz * escala
    J = []
    for _ in range(N):
        k = rng.choice([1, 2, 3, 4], p=P_TOQUES)
        seq = [rng.choice(canales, p=peso)]
        for _ in range(k - 1):
            seq.append(seq[-1] if rng.random() < P_REPETIR else rng.choice(canales, p=peso))
        t = np.concatenate([[0], np.cumsum(rng.exponential(GAP_DIAS, k - 1))])
        h = hz[seq].to_numpy()
        p = 1 - np.exp(-h.sum())
        J.append({"seq": seq, "t": t, "h": h, "p": p, "conv": rng.random() < p})
    return J

def atribuir(J, metodo):
    cred = defaultdict(float)
    for j in J:
        s, t = j["seq"], j["t"]
        if metodo == "Verdad":
            w = j["p"] * j["h"] / j["h"].sum()
        elif not j["conv"]:
            continue
        elif metodo == "First-touch":
            w = np.eye(len(s))[0]
        elif metodo == "Last-touch":
            w = np.eye(len(s))[-1]
        elif metodo == "Lineal":
            w = np.ones(len(s)) / len(s)
        elif metodo == "Decaimiento":
            w = 0.5 ** ((t[-1] - t) / 7)
            w = w / w.sum()
        for canal, wi in zip(s, w):
            cred[canal] += wi
    r = pd.Series(cred)
    return r / r.sum()

# 1. Calibrar la escala del riesgo (bisección)
lo, hi = 0.1, 1.5
for _ in range(12):
    mid = (lo + hi) / 2
    pm = np.mean([j["p"] for j in simular(0, N=5000, escala=mid)])
    lo, hi = (mid, hi) if pm < objetivo else (lo, mid)
escala = (lo + hi) / 2
print(f"escala = {escala:.3f} | conversión objetivo = {objetivo:.4f}")

# 2. Repetir con 10 semillas
metodos = ["First-touch", "Last-touch", "Lineal", "Decaimiento", "Markov"]
maes = []
for seed in range(10):
    J = simular(seed, escala=escala)
    journeys = [["Inicio", *j["seq"], "Conversion" if j["conv"] else "Sin_conversion"] for j in J]
    comp = pd.DataFrame({m: atribuir(J, m) for m in metodos[:-1]})
    comp["Markov"] = efecto_remocion(journeys)
    comp["Verdad"] = atribuir(J, "Verdad")
    comp = comp.fillna(0)
    maes.append(comp[metodos].sub(comp["Verdad"], axis=0).abs().mean() * 100)

maes = pd.DataFrame(maes)
print("conversión simulada (última semilla):", round(np.mean([j["conv"] for j in J]), 4))
print("\nMAE vs verdad (pp), 10 semillas:")
print(pd.DataFrame({"media": maes.mean(), "desv": maes.std()}).round(3).sort_values("media"))
print("\nÚltima semilla:\n", comp.round(4).sort_values("Verdad", ascending=False))

escala = 0.516 | conversión objetivo = 0.1130
conversión simulada (última semilla): 0.1132

MAE vs verdad (pp), 10 semillas:
             media   desv
Lineal       1.231  0.090
Decaimiento  1.254  0.109
First-touch  1.285  0.093
Last-touch   1.326  0.168
Markov       2.276  0.113

Última semilla:
                    First-touch  Last-touch  Lineal  Decaimiento  Markov  \
organic_search          0.3188      0.3210  0.3185       0.3191  0.2896   
unknown                 0.2026      0.2062  0.2082       0.2075  0.1853   
paid_search             0.1991      0.2018  0.2005       0.2004  0.1989   
social                  0.1152      0.1095  0.1111       0.1108  0.1359   
direct_traffic          0.0702      0.0640  0.0667       0.0654  0.0681   
referral                0.0349      0.0331  0.0340       0.0334  0.0368   
email                   0.0353      0.0318  0.0346       0.0343  0.0462   
display                 0.0110      0.0110  0.0104       0.0109  0.0135   
other                   0.

In [4]:
maes.agg(["mean", "std"]).T.round(3).to_csv("../data/resultados_mae.csv")
comp.round(4).to_csv("../data/atribucion_simulada.csv")